# SMS Spam Detection - Exploratory Data Analysis and Model Evaluation

This notebook covers loading the dataset, cleaning the text, vectorizing it, and comparing two models: Naive Bayes and Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, confusion_matrix

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Load and Clean the Data

In [ ]:
# Load dataset
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
print(df.head())

sns.countplot(x='label', data=df)
plt.title('Distribution of Spam vs Ham')
plt.show()

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_message'] = df['message'].apply(clean_text)
df.head()

## 2. Vectorize Text and Train-Test Split

In [ ]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Training features shape: {X_train_vec.shape}")

## 3. Train and Compare Models

In [ ]:
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label='spam')
    rec = recall_score(y_true, y_pred, pos_label='spam')
    cm = confusion_matrix(y_true, y_pred, labels=['ham', 'spam'])
    
    print(f"\n{name} Evaluation:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    plt.title(f"{name} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
nb_pred = nb_model.predict(X_test_vec)
evaluate_model("Naive Bayes", y_test, nb_pred)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)
lr_pred = lr_model.predict(X_test_vec)
evaluate_model("Logistic Regression", y_test, lr_pred)

## 4. Model Limitations and Improvements

### Limitations
- **Short Messages**: The model struggles with extremely short messages like "ok" or "yes" if they haven't appeared frequently with a specific label.
- **Unseen Slang/Misspellings**: Spammers constantly evolve their vocabulary. If new slang or intentional misspellings are used (e.g., "f.r.e.e"), standard TF-IDF might not capture it well because it relies on exact string matches (unless character n-grams are used).
- **Sarcasm and Context**: Simple models like Naive Bayes and Logistic Regression lack an understanding of sentence structure, sequence, or deeper context (sarcasm). They look purely at word frequency/presence.

### Improvements
- **N-Grams**: Incorporating bi-grams or tri-grams (e.g., `ngram_range=(1,2)`) in the TfidfVectorizer would capture short phrases (e.g., "win cash").
- **Advanced Models**: Using a transformer-based model like BERT or a deep learning LSTM model could capture semantic meaning and sequence much better.
- **Feature Engineering**: Adding meta-features such as message length, count of uppercase letters, count of special characters (like `$` or `!`), or the presence of a URL could significantly boost the model's predictive power without changing the algorithm.